# Explore the SkyWays world

Run every cell in order. By the end you will have traced one driver message from the chat table to the dock slot it threatens, and you will know which tables your build has to touch.

Run from the kit root so the relative path below works.

In [ ]:
import sqlite3, pandas as pd
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 200)
con = sqlite3.connect('data/skyways_freight_operations.db')
con.execute('PRAGMA foreign_keys = ON')
q = lambda sql, *p: pd.read_sql_query(sql, con, params=p)
q("select name, type from sqlite_master where type in ('table','view') order by type, name")

## 1. The snapshot moment

The database is frozen at one moment: 4 August 2026, around 10:30 in the morning at Jaipur. Every question you ask is 'what is true right now'.

In [ ]:
q("select facility_id, facility_name, open_time, close_time, checkin_grace_min, default_unload_min from facilities")

In [ ]:
q("select * from v_inbound_operational_state where destination_facility_id='FAC-JAI-01' order by effective_eta_ts")

## 2. One message, traced

Thread THR001 is a driver stuck in traffic. Follow it through four tables and write down, in the markdown cell after, which columns you needed.

In [ ]:
q("select chat_message_id, sender_type, message_text, message_ts, is_duplicate, requires_human_review from chat_messages where thread_id='THR001' order by message_ts")

In [ ]:
q("select * from chat_threads where thread_id='THR001'")

In [ ]:
q("select * from driver_exceptions where thread_id='THR001'")

In [ ]:
shp = q("select shipment_id from chat_threads where thread_id='THR001'").iloc[0,0]
q("select * from eta_updates where shipment_id=? order by created_at", shp)

In [ ]:
q("select a.*, s.slot_start_ts, s.slot_end_ts, s.dock_id from appointments a join appointment_slots s on s.slot_id=a.slot_id where a.shipment_id=? order by a.booked_at", shp)

**Your notes:** which columns did you need to answer 'is the current appointment still feasible'? Write them here.

- 
- 
- 

## 3. What does 'available' mean?

A slot can be OPEN in `appointment_slots` and still carry an active appointment. The view `v_slot_availability` resolves this. Compare the two.

In [ ]:
q("select slot_status, count(*) from appointment_slots where facility_id='FAC-JAI-01' group by slot_status")

In [ ]:
q("select availability_status, count(*) from v_slot_availability where facility_id='FAC-JAI-01' group by availability_status")

In [ ]:
q("""select slot_id, dock_code, dock_type, slot_start_ts, slot_end_ts, availability_status
     from v_slot_availability where facility_id='FAC-JAI-01' and dock_type='STANDARD'
       and availability_status='AVAILABLE' and slot_start_ts >= '2026-08-04T11:20:00+05:30'
     order by slot_start_ts, dock_code limit 12""")

## 4. The competition

Four delayed trucks want standard docks around midday. List them with priority and unload minutes, then answer in the cell below: if only two slots exist before 13:00, who should get them and why? There is no single right answer; there is a defensible one.

In [ ]:
q("""select e.exception_id, e.shipment_id, e.declared_eta_ts, s.priority_code, s.required_dock_type, s.expected_unload_min, e.exception_status
     from driver_exceptions e join shipments s on s.shipment_id=e.shipment_id
     where e.exception_status in ('OPEN','NEEDS_INFORMATION','WAITING_CONFIRMATION') order by e.declared_eta_ts""")

**Your allocation and the rule behind it:**



## 5. The case with no answer

SHP1015 is a reefer load whose only compatible dock goes into maintenance. Prove to yourself there is no feasible slot tonight.

In [ ]:
q("select * from dock_status_events")

In [ ]:
q("""select slot_id, dock_code, slot_start_ts, availability_status from v_slot_availability
     where facility_id='FAC-JAI-01' and dock_type='REEFER' and slot_start_ts >= '2026-08-04T18:00:00+05:30' order by slot_start_ts""")

## 6. The history file (ML and Hybrid tracks)

Six thousand past visits. Look at the shape, the missing values and the duplicates before you believe any number.

In [ ]:
h = pd.read_csv('data/dock_visit_history.csv')
print(h.shape); print(h.visit_outcome.value_counts()); print('duplicates:', h.duplicated().sum()); h.isna().sum()[lambda s: s>0]

In [ ]:
h.groupby('dock_type')['unload_min'].describe().round(1)

## 7. The driver messages (Agentic and Hybrid tracks)

148 labelled messages. Read the ones that need clarification and ask: what single question would resolve each?

In [ ]:
m = pd.read_csv('data/driver_messages_labelled.csv')
print(m.intent.value_counts())
m[m.needs_clarification==1][['message_text','intent','clarification_reason']]

## Before you close this notebook

Write three candidate pain points in `01_discover/pain_point_depth_matrix.md`. Each must name a table you saw here.